In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, mean_squared_error
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# 1. Define Paths and Create Models Directory
base_path = r"C:\Users\Jam Villanueva\Documents\AIM PGD AIML\1_Capstone Project"
data_path = os.path.join(base_path, "data", "synthetic_brownfield_mto_data.csv")
models_dir = os.path.join(base_path, "models")
os.makedirs(models_dir, exist_ok=True)

# 2. Load and Prep Data (Re-applying the Step 3 pipeline quickly)
df = pd.read_csv(data_path)
df.drop_duplicates(inplace=True)

for col in ['MTO_Growth_Pct', 'Final_AsBuilt_Qty']:
    df[col] = np.where(df[col] > df[col].quantile(0.99), df[col].quantile(0.99), df[col])

df['FEED_Gate_Maturity_Encoded'] = df['FEED_Gate_Maturity'].map({'30%': 1, '60%': 2, '90%': 3})
df_encoded = pd.get_dummies(df, columns=['Metallurgy', 'Fluid_Class'], drop_first=True)
df_encoded['Process_Severity_Index'] = df_encoded['Design_Pressure_barg'] * df_encoded['Design_Temp_C']
df_encoded['Brownfield_Complexity_Penalty'] = df_encoded['Tie_In_Density'] / (df_encoded['Laser_Scan_Coverage_Pct'] + 1)

# 3. Define Targets, Groups, and Features
groups = df_encoded['Project_ID'] # Used for Group K-Fold
y_class = df_encoded['High_Risk_Flag']
y_reg = df_encoded['MTO_Growth_Pct']
X = df_encoded.drop(columns=['Project_ID', 'Line_Tag', 'FEED_Gate_Maturity', 'High_Risk_Flag', 'MTO_Growth_Pct', 'Final_AsBuilt_Qty', 'FEED_MTO_Qty'])

# 4. Scale Features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Save the scaler for future inference
joblib.dump(scaler, os.path.join(models_dir, 'standard_scaler.pkl'))
print("Data prepped and Scaler saved. Ready for modeling.")

Data prepped and Scaler saved. Ready for modeling.


In [2]:
# Initialize Classification Models (Memory-safe parameters)
class_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=2),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=2),
    'LightGBM': lgb.LGBMClassifier(random_state=42, n_jobs=2, verbose=-1)
}

# Group K-Fold Cross Validation (5 splits)
gkf = GroupKFold(n_splits=5)
class_results = {name: [] for name in class_models.keys()}

print("--- Classification Results (PR-AUC) ---")
for name, model in class_models.items():
    fold_scores = []
    for train_idx, val_idx in gkf.split(X_scaled, y_class, groups):
        X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
        y_train, y_val = y_class.iloc[train_idx], y_class.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        
        # Calculate Precision-Recall AUC
        score = average_precision_score(y_val, y_pred_proba)
        fold_scores.append(score)
        
    avg_score = np.mean(fold_scores)
    class_results[name] = avg_score
    print(f"{name}: {avg_score:.4f}")
    
    # Save the final model trained on full dataset
    model.fit(X_scaled, y_class)
    joblib.dump(model, os.path.join(models_dir, f"{name.replace(' ', '_').lower()}_classifier.pkl"))

--- Classification Results (PR-AUC) ---
Logistic Regression: 0.9719
Random Forest: 0.9668
XGBoost: 0.9703
LightGBM: 0.9741


In [3]:
# Initialize Regression Models
reg_models = {
    'Ridge Regression': Ridge(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2),
    'XGBoost': xgb.XGBRegressor(random_state=42, eval_metric='rmse', n_jobs=2),
    'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=2, verbose=-1)
}

reg_results = {name: [] for name in reg_models.keys()}

print("\n--- Regression Results (RMSE) ---")
for name, model in reg_models.items():
    fold_scores = []
    for train_idx, val_idx in gkf.split(X_scaled, y_reg, groups):
        X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
        y_train, y_val = y_reg.iloc[train_idx], y_reg.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        # Calculate Root Mean Squared Error
        score = np.sqrt(mean_squared_error(y_val, y_pred))
        fold_scores.append(score)
        
    avg_score = np.mean(fold_scores)
    reg_results[name] = avg_score
    print(f"{name}: {avg_score:.4f}")
    
    # Save the final model
    model.fit(X_scaled, y_reg)
    joblib.dump(model, os.path.join(models_dir, f"{name.replace(' ', '_').lower()}_regressor.pkl"))

print(f"\nAll models and scalers successfully saved to: {models_dir}")


--- Regression Results (RMSE) ---
Ridge Regression: 3.7979
Random Forest: 3.7756
XGBoost: 3.7176
LightGBM: 3.5090

All models and scalers successfully saved to: C:\Users\Jam Villanueva\Documents\AIM PGD AIML\1_Capstone Project\models
